Chains in Langchain

In [2]:
from langchain_openai import ChatOpenAI
import os

llm_openai = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model="nvidia/nemotron-nano-9b-v2:free",
    temperature=0
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage,AIMessage
from langchain_core.output_parsers import StrOutputParser


# *Chain with Conditional Chains*

In [16]:
# Task 1

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie review Evaluator"),
        ("human","please Evaluate the movie review is negative or Positive :{input}")
    ])

In [5]:
#Task 2

llm_openai = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model="nvidia/nemotron-nano-9b-v2:free",
    temperature=0
)


In [ ]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag:Literal["Positive","Negative"]

llm_structured_output=llm_openai.with_structured_output(llm_schema)


llm_schema(movie_summary_flag='Positive')

In [17]:
# Task 4

from langchain_core.runnables import RunnableLambda

def pydantic_json(input: llm_schema) -> str:
        return input.model_dump()['movie_summary_flag']

pydantic_json_runnable = RunnableLambda(pydantic_json)

# *Conditional Chain 1*

In [13]:
linkedIn_post = ChatPromptTemplate.from_messages(
    messages = [        
    ("system","you are a linkedIn post generator "),
    ("human","Create a post for the LinkedIn for the text {text}")
    ]
    )

llm_openai = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model="nvidia/nemotron-nano-9b-v2:free",
    temperature=0
)


str_parser = StrOutputParser()

linkedin_chain_runnable = linkedIn_post | llm_openai | str_parser

# *Conditional Chain 2*

In [8]:
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnableBranch

In [42]:
def insta_post(text:str):
    Instagram_post = ChatPromptTemplate.from_messages(
    messages = [        
            ("system","you are a Instagram post generator "),
            ("human","Create a post for the Instagram for the text {text}")
            ]
        )

    llm_openai = ChatOpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ.get("OPENROUTER_API_KEY"),
            model="nvidia/nemotron-nano-9b-v2:free",
            temperature=0
        )

    str_parser = StrOutputParser()

    chain_insta = Instagram_post | llm_openai | str_parser

    return chain_insta.invoke({"text":text})

insta_chain_runnable = RunnableLambda(insta_post)

# *Final Orchestration*


In [43]:
conditional_chain = RunnableBranch(
  (lambda x: "Positive" in x,linkedin_chain_runnable),
  insta_chain_runnable
)


In [45]:

final_orchastrator = prompt |  llm_structured_output | pydantic_json_runnable | conditional_chain

final_orchastrator.invoke({"input" : "I love the movie"})

'\n\nCertainly! Here\'sa professional and engaging LinkedIn post based on the theme of "Positive." You can customize it further depending on your specific context (e.g., personal achievement, company news, or general inspiration):\n\n---\n\n**🌟 Embracing Positivity: A Mindset That Transforms** 🌟  \n\nIn a world that often focuses on challenges, I’ve always believed that **positivity** is a powerful force. It’s not just about ignoring the negatives—it’s about choosing to focus on what *can* be, what *is*, and what *will* be.  \n\nWhether it’s a small win, a kind gesture, or a moment of gratitude, **positivity** has the power to uplift not just ourselves, but those around us. It fuels resilience, fosters connection, and opens doors to opportunities we might not have seen otherwise.  \n\n**Why positivity matters:**  \n✅ It shapes our perspective.  \n✅ It attracts positivity (yes, really!).  \n✅ It’s a choice we make every day.  \n\nSo, let’s celebrate the good, learn from the tough, and k